In [1]:
import pandas as pd
import numpy as np

sysarmy = pd.read_csv('../data/raw/sysarmy_2025_2.csv', skiprows=9)
indec   = pd.read_csv('../data/raw/serie_ipc_divisiones.csv', encoding='latin-1', sep=';')
stackoverflow = pd.read_csv('../data/raw/datosInternacionales.csv')

print("Datasets cargados")

C:\Users\benja\AppData\Local\Temp\ipykernel_23972\1752325327.py:6: DtypeWarning: Columns (0: JobSatPoints_15_TEXT, 1: DatabaseHaveEntry, 2: DevEnvHaveEntry, 3: SOTagsHaveEntry, 4: SOTagsWant Entry, 5: OfficeStackWantEntry, 6: CommPlatformHaveEntr, 7: CommPlatformWantEntr, 8: SO_Actions_15_TEXT, 9: AIAgentOrchestration, 10: AIAgentObsWrite) have mixed types. Specify dtype option on import or set low_memory=False.
  stackoverflow = pd.read_csv('../data/raw/datosInternacionales.csv')


Datasets cargados


In [2]:
print("=== NULOS SYSARMY ===")
nulos = sysarmy.isnull().sum()
total = len(sysarmy)

nulos_df = pd.DataFrame({
    'nulos': nulos,
    'porcentaje': (nulos / total * 100).round(1)
})

print(nulos_df[nulos_df['nulos'] > 0].sort_values('porcentaje', ascending=False))

=== NULOS SYSARMY ===
                                                    nulos  porcentaje
pluriempleo                                          3516        93.8
si_tu_sueldo_esta_dolarizado_cual_fue_el_ultimo...   2939        78.4
aclara_el_numero_que_ingresaste_en_el_campo_ant...   2788        74.4
tenes_guardias                                       2788        74.4
cuanto_cobras_por_guardia                            2788        74.4
desafios_tecnicoscomo_los_preferiselegi_todas_l...   2607        69.6
cuantas_horas_te_parecen_correctas_dedicarle_a_...   2600        69.4
cual_de_las_siguientes_frases_representa_mejor_...   2599        69.3
salir_o_seguir_contestando_sobre_las_guardias1       2578        68.8
cual_es_la_cantidad_maxima_de_entrevistas_que_c...   2578        68.8
cual_es_la_combinacion_de_entrevistas_que_consi...   2579        68.8
institucion_educativa                                2539        67.7
pagos_en_dolares                                     2478        66.

In [3]:
col_salario = 'ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos'

print(f"Nulos: {sysarmy[col_salario].isnull().sum()}")
print(f"Ceros: {(sysarmy[col_salario] == 0).sum()}")
print(f"\nEstadísticas:")
print(sysarmy[col_salario].describe())

Nulos: 0
Ceros: 0

Estadísticas:
count    3.748000e+03
mean     3.317345e+06
std      1.986262e+06
min      1.700000e+05
25%      1.936722e+06
50%      2.899000e+06
75%      4.200000e+06
max      1.567500e+07
Name: ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos, dtype: float64


In [9]:
sysarmy_clean = sysarmy_clean.rename(columns={'pais': 'provincia'})

# Recuperar las filas que eliminamos por el filtro incorrecto
sysarmy_clean = sysarmy[columnas_utiles].copy()
sysarmy_clean.columns = [
    'provincia', 'dedicacion', 'tipo_contrato', 'salario_bruto', 'salario_neto',
    'pagos_en_dolares', 'tiene_bono', 'rol', 'anos_experiencia', 'antiguedad_empresa',
    'anos_puesto', 'tecnologias', 'frameworks', 'bases_de_datos', 'modalidad',
    'tamano_empresa', 'nivel_estudios', 'genero', 'seniority', 'sueldo_dolarizado'
]

# Volver a aplicar solo el filtro de outliers
q01 = sysarmy_clean['salario_bruto'].quantile(0.01)
q99 = sysarmy_clean['salario_bruto'].quantile(0.99)
sysarmy_clean = sysarmy_clean[
    (sysarmy_clean['salario_bruto'] >= q01) & 
    (sysarmy_clean['salario_bruto'] <= q99)
]

print(f"Filas finales: {len(sysarmy_clean)}")
print(f"Rango salario: ${q01:,.0f} — ${q99:,.0f}")
sysarmy_clean['provincia'].value_counts().head(10)

Filas finales: 3676
Rango salario: $600,000 — $10,000,000


provincia
Ciudad Autónoma de Buenos Aires    1830
Buenos Aires                        761
Córdoba                             328
Santa Fe                            273
Mendoza                             120
Entre Ríos                           61
Río Negro                            37
Misiones                             33
Tucumán                              32
Chaco                                28
Name: count, dtype: int64

In [10]:
### Celda 8 — Nulos restantes
print("=== NULOS RESTANTES ===")
nulos = sysarmy_clean.isnull().sum()
print(nulos[nulos > 0])

=== NULOS RESTANTES ===
salario_neto         161
pagos_en_dolares    2443
tecnologias            1
frameworks             2
bases_de_datos         1
nivel_estudios      2185
dtype: int64


In [11]:
### Celda 9 — Limpiar nulos
# Columnas de texto: rellenar con 'No especifica'
cols_texto = ['pagos_en_dolares', 'tiene_bono', 'modalidad', 
              'nivel_estudios', 'genero', 'seniority', 'sueldo_dolarizado']

for col in cols_texto:
    sysarmy_clean[col] = sysarmy_clean[col].fillna('No especifica')

# Columnas numéricas: rellenar con mediana
cols_numericas = ['salario_neto', 'anos_experiencia', 
                  'antiguedad_empresa', 'anos_puesto']

for col in cols_numericas:
    mediana = sysarmy_clean[col].median()
    sysarmy_clean[col] = sysarmy_clean[col].fillna(mediana)

# Columnas de tecnologías: rellenar con vacío
cols_tech = ['tecnologias', 'frameworks', 'bases_de_datos']
for col in cols_tech:
    sysarmy_clean[col] = sysarmy_clean[col].fillna('No especifica')

print("=== NULOS DESPUÉS DE LIMPIEZA ===")
print(sysarmy_clean.isnull().sum().sum(), "nulos totales restantes")

=== NULOS DESPUÉS DE LIMPIEZA ===
0 nulos totales restantes


In [12]:
### Celda 10 — Normalizar texto en columnas clave
# Minúsculas y sin espacios extra
cols_normalizar = ['rol', 'seniority', 'modalidad', 'genero', 'provincia']

for col in cols_normalizar:
    sysarmy_clean[col] = sysarmy_clean[col].str.strip().str.lower()

# Ver valores únicos de seniority para verificar
print("=== SENIORITY ===")
print(sysarmy_clean['seniority'].value_counts())

print("\n=== MODALIDAD ===")
print(sysarmy_clean['modalidad'].value_counts())

=== SENIORITY ===
seniority
senior         2029
semi-senior    1198
junior          449
Name: count, dtype: int64

=== MODALIDAD ===
modalidad
100% remoto                      1811
híbrido (presencial y remoto)    1586
100% presencial                   279
Name: count, dtype: int64


In [13]:
### Celda 11 — Guardar sysarmy limpio
sysarmy_clean.to_csv('../data/processed/sysarmy_limpio.csv', index=False)
print(f"Guardado: {len(sysarmy_clean)} filas, {sysarmy_clean.shape[1]} columnas")

Guardado: 3676 filas, 20 columnas


In [14]:
### Celda 12 — Limpiar INDEC
# Ver qué valores tiene la columna Descripcion
print("=== DESCRIPCIONES INDEC ===")
print(indec['Descripcion'].value_counts())

=== DESCRIPCIONES INDEC ===
Descripcion
NIVEL GENERAL                                             791
Alimentos y bebidas no alcohólicas                        791
Bebidas alcohólicas y tabaco                              791
Prendas de vestir y calzado                               791
Vivienda, agua, electricidad, gas y otros combustibles    791
Equipamiento y mantenimiento del hogar                    791
Salud                                                     791
Transporte                                                791
Comunicación                                              791
Recreación y cultura                                      791
Educación                                                 791
Restaurantes y hoteles                                    791
Bienes y servicios varios                                 791
Name: count, dtype: int64


In [15]:
### Celda 13 — Filtrar INDEC
indec_clean = indec[indec['Descripcion'] == 'NIVEL GENERAL'].copy()

# Convertir Periodo a string para manipular
indec_clean['Periodo'] = indec_clean['Periodo'].astype(str)

# Filtrar desde 2020 en adelante
indec_clean = indec_clean[indec_clean['Periodo'] >= '202001']

print(f"Filas: {len(indec_clean)}")
print(f"Período: {indec_clean['Periodo'].min()} → {indec_clean['Periodo'].max()}")
print(f"\nRegiones disponibles:")
print(indec_clean['Region'].value_counts())

Filas: 532
Período: 202001 → 202604

Regiones disponibles:
Region
GBA          76
Pampeana     76
Noreste      76
Noroeste     76
Cuyo         76
Patagonia    76
Nacional     76
Name: count, dtype: int64


In [16]:
### Celda 14 — Quedarnos con el promedio nacional
indec_clean = indec_clean[indec_clean['Region'] == 'Nacional'].copy()

# Quedarnos solo con las columnas útiles
indec_clean = indec_clean[['Periodo', 'Indice_IPC', 'v_m_IPC', 'v_i_a_IPC']].copy()

# Renombrar
indec_clean.columns = ['periodo', 'indice_ipc', 'var_mensual', 'var_interanual']

print(f"Filas finales: {len(indec_clean)}")
indec_clean.tail(10)

Filas finales: 76


,periodo,indice_ipc,var_mensual,var_interanual
12984,202507,"9023,973","1,9","36,6"
13110,202508,"9193,2441","1,9","33,6"
13236,202509,"9384,0922","2,1","31,8"
13362,202510,"9603,8623","2,3","31,3"
13488,202511,"9841,3581","2,5","31,4"
13614,202512,"10121,3715","2,8","31,5"
13740,202601,"10413,0309","2,9","32,4"
13866,202602,"10714,6255","2,9","33,1"
13992,202603,"11077,0608","3,4","32,6"
14118,202604,"11363,0904","2,6","32,4"


In [17]:
### Celda 15 — Guardar INDEC limpio
indec_clean.to_csv('../data/processed/indec_limpio.csv', index=False)
print("INDEC guardado")

INDEC guardado


In [ ]:
### Celda 16 — Corregir formato numérico del INDEC
indec_clean = pd.read_csv('../data/processed/indec_limpio.csv', dtype=str)

# Reemplazar coma por punto en columnas numéricas
cols_numericas = ['indice_ipc', 'var_mensual', 'var_interanual']
for col in cols_numericas:
    indec_clean[col] = indec_clean[col].str.replace(',', '.').astype(float)

# Periodo como string limpio
indec_clean['periodo'] = indec_clean['periodo'].astype(str)

print("Formato corregido")
print(indec_clean.dtypes)
print()
indec_clean.tail(3)

✅ Formato corregido
periodo               str
indice_ipc        float64
var_mensual       float64
var_interanual    float64
dtype: object



,periodo,indice_ipc,var_mensual,var_interanual
73,202602,10714.6255,2.9,33.1
74,202603,11077.0608,3.4,32.6
75,202604,11363.0904,2.6,32.4


In [19]:
### Celda 17 — Guardar INDEC corregido
indec_clean.to_csv('../data/processed/indec_limpio.csv', index=False)
print("INDEC guardado con formato correcto")

INDEC guardado con formato correcto


In [20]:
### Celda 18 — Explorar Stack Overflow
print("=== COLUMNAS RELEVANTES ===")
cols_relevantes = [
    'ResponseId', 'Age', 'EdLevel', 'Employment', 'WorkExp',
    'DevType', 'OrgSize', 'RemoteWork', 'YearsCode',
    'Country', 'ConvertedCompYearly', 'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith', 'FrameworkHaveWorkedWith'
]

# Verificar que todas existan
for col in cols_relevantes:
    existe = col in stackoverflow.columns
    print(f"  {'✅' if existe else '❌'} {col}")

=== COLUMNAS RELEVANTES ===
  ✅ ResponseId
  ✅ Age
  ✅ EdLevel
  ✅ Employment
  ✅ WorkExp
  ✅ DevType
  ✅ OrgSize
  ✅ RemoteWork
  ✅ YearsCode
  ✅ Country
  ✅ ConvertedCompYearly
  ✅ LanguageHaveWorkedWith
  ✅ DatabaseHaveWorkedWith
  ❌ FrameworkHaveWorkedWith


In [22]:
### Celda 19 — Filtrar columnas y filas con salario (corregido)
cols_relevantes = [
    'ResponseId', 'Age', 'EdLevel', 'Employment', 'WorkExp',
    'DevType', 'OrgSize', 'RemoteWork', 'YearsCode',
    'Country', 'ConvertedCompYearly', 'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith'
]

so_clean = stackoverflow[cols_relevantes].copy()

# Quedarse solo con los que tienen salario informado
antes = len(so_clean)
so_clean = so_clean[so_clean['ConvertedCompYearly'].notna()]
despues = len(so_clean)

print(f"Filas con salario informado: {despues} (eliminadas: {antes - despues})")
print(f"\nTop 10 países:")
print(so_clean['Country'].value_counts().head(10))

Filas con salario informado: 23947 (eliminadas: 25244)

Top 10 países:
Country
United States of America                                5243
Germany                                                 2141
United Kingdom of Great Britain and Northern Ireland    1486
India                                                   1096
France                                                  1027
Canada                                                   915
Ukraine                                                  709
Brazil                                                   639
Netherlands                                              619
Poland                                                   589
Name: count, dtype: int64


In [23]:
print(so_clean['Country'].value_counts().get('Argentina', 0), "respuestas de Argentina")
print(so_clean['Country'].value_counts().get('Latin America', 0))

# Buscar variantes del nombre
paises = so_clean['Country'].unique()
argentina = [p for p in paises if 'rgentin' in str(p)]
print(f"Variantes encontradas: {argentina}")

160 respuestas de Argentina
0
Variantes encontradas: ['Argentina']


In [24]:
### Celda 20 — Renombrar columnas y limpiar
so_clean.columns = [
    'id', 'edad', 'nivel_estudios', 'empleo', 'anos_experiencia',
    'rol', 'tamano_empresa', 'modalidad', 'anos_codigo',
    'pais', 'salario_anual_usd', 'tecnologias', 'bases_de_datos'
]

# Eliminar outliers de salario (top 1%)
q99 = so_clean['salario_anual_usd'].quantile(0.99)
q01 = so_clean['salario_anual_usd'].quantile(0.01)
antes = len(so_clean)
so_clean = so_clean[
    (so_clean['salario_anual_usd'] >= q01) &
    (so_clean['salario_anual_usd'] <= q99)
]
print(f"Filas eliminadas por outliers: {antes - len(so_clean)}")
print(f"Rango salario válido: USD {q01:,.0f} — USD {q99:,.0f}")
print(f"Filas finales: {len(so_clean)}")

Filas eliminadas por outliers: 473
Rango salario válido: USD 65 — USD 440,856
Filas finales: 23474


In [25]:
### Celda 21 — Crear subsets útiles para el análisis
# Dataset solo Argentina
so_argentina = so_clean[so_clean['pais'] == 'Argentina'].copy()

# Dataset Latinoamérica para contexto
latam = ['Argentina', 'Brazil', 'Mexico', 'Colombia', 'Chile', 
         'Uruguay', 'Peru', 'Venezuela, Bolivarian Republic of...']
so_latam = so_clean[so_clean['pais'].isin(latam)].copy()

print(f"Stack Overflow Argentina: {len(so_argentina)} respuestas")
print(f"Stack Overflow LATAM: {len(so_latam)} respuestas")

Stack Overflow Argentina: 149 respuestas
Stack Overflow LATAM: 1282 respuestas


In [26]:
### Celda 22 — Guardar los 3 subsets
so_clean.to_csv('../data/processed/stackoverflow_limpio.csv', index=False)
so_argentina.to_csv('../data/processed/stackoverflow_argentina.csv', index=False)
so_latam.to_csv('../data/processed/stackoverflow_latam.csv', index=False)

print("Stack Overflow guardado en 3 versiones:")
print(f"  - stackoverflow_limpio.csv      ({len(so_clean)} filas) — global")
print(f"  - stackoverflow_argentina.csv   ({len(so_argentina)} filas) — solo AR")
print(f"  - stackoverflow_latam.csv       ({len(so_latam)} filas) — LATAM")

Stack Overflow guardado en 3 versiones:
  - stackoverflow_limpio.csv      (23474 filas) — global
  - stackoverflow_argentina.csv   (149 filas) — solo AR
  - stackoverflow_latam.csv       (1282 filas) — LATAM
